In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42

print("=" * 60)
print("09 — MODEL EXPORT ve FEDERATED HAZIRLIK")
print("=" * 60)

09 — MODEL EXPORT ve FEDERATED HAZIRLIK


In [2]:
import re

FEATURES_TXT       = Path("../data/csv/selected_features.txt")
FEATURES_CLEAN_TXT = Path("../data/csv/selected_features_clean.txt")

with open(FEATURES_TXT) as f:
    original_features = [line.strip() for line in f if line.strip()]

print(f"Orijinal feature sayısı: {len(original_features)}")

# 1) Leakage feature'ları çıkar
LEAKAGE_PATTERNS = [
    r"source.?address", r"destination.?address",
    r"src.?ip", r"dst.?ip",
    r"source.?ip", r"destination.?ip",
    r"flow.?id", r"timestamp",
]
PORT_PATTERNS = [r"^destination.?port$"]

def find_matching_columns(columns, patterns):
    found = []
    for col in columns:
        col_norm = col.lower().strip().replace(" ", "_")
        for pat in patterns:
            if re.search(pat, col_norm):
                found.append(col)
                break
    return sorted(set(found))

leak_found = find_matching_columns(original_features, LEAKAGE_PATTERNS)
port_found = find_matching_columns(original_features, PORT_PATTERNS)
all_leak   = sorted(set(leak_found + port_found))

clean_features = [f for f in original_features if f not in all_leak]
print(f"Leakage olarak çıkarılan: {all_leak}")

# 2) min_seg_size_forward çıkar
TARGET = "min_seg_size_forward"
if TARGET in clean_features:
    clean_features.remove(TARGET)
    print(f"Skewness nedeniyle çıkarılan: ['{TARGET}']")
else:
    print(f"'{TARGET}' zaten listede yok.")

# Kaydet
with open(FEATURES_CLEAN_TXT, "w") as f:
    for feat in sorted(clean_features):
        f.write(feat + "\n")

print(f"\nKalan feature sayısı: {len(clean_features)}")
print(f"✓ Kaydedildi: {FEATURES_CLEAN_TXT}")

Orijinal feature sayısı: 47
Leakage olarak çıkarılan: ['Destination Port']
Skewness nedeniyle çıkarılan: ['min_seg_size_forward']

Kalan feature sayısı: 45
✓ Kaydedildi: ../data/csv/selected_features_clean.txt


In [3]:
FEATURED_PATH  = Path("../data/csv/featured_dataset.csv")
LABEL_MAP_PATH = Path("../data/csv/label_mapping.csv")
MODELS_DIR     = Path("../models")
FEDERATED_DIR  = Path("../federated")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FEDERATED_DIR.mkdir(parents=True, exist_ok=True)

# Temizlenmiş feature listesini oku
with open(FEATURES_CLEAN_TXT) as f:
    CLEAN_FEATURES = [line.strip() for line in f if line.strip()]

# RAM dostu: sadece gerekli kolonları oku
needed_cols = CLEAN_FEATURES + ["label_binary", "label_multiclass"]
df = pd.read_csv(FEATURED_PATH, usecols=lambda c: c in needed_cols, low_memory=False)

# Eksik feature kontrolü
missing_feats = [f for f in CLEAN_FEATURES if f not in df.columns]
if missing_feats:
    print(f"[UYARI] Eksik feature kolonları: {missing_feats}")
    CLEAN_FEATURES = [f for f in CLEAN_FEATURES if f in df.columns]

X = df[CLEAN_FEATURES].fillna(0)

print("=" * 60)
print("FINAL MODEL EĞİTİMİ")
print("=" * 60)
print(f"Feature sayısı: {len(CLEAN_FEATURES)}")
print()

# Her iki label türü için model eğit
for label_col, pkl_name in [
    ("label_multiclass", "rf_multiclass.pkl"),
    ("label_binary",     "rf_binary.pkl")
]:
    if label_col not in df.columns:
        print(f"[UYARI] {label_col} kolonu yok, atlanıyor.")
        continue

    y = df[label_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=2
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    f1_m = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"--- {label_col} ---")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  F1 Weighted : {f1_w:.4f}")
    print(f"  F1 Macro    : {f1_m:.4f}")

    # Model + feature listesini birlikte kaydet
    bundle = {
        "model":    model,
        "features": CLEAN_FEATURES,
        "label_col": label_col,
    }
    out_path = MODELS_DIR / pkl_name
    joblib.dump(bundle, out_path)
    print(f"  ✓ Kaydedildi: {out_path}")

    # Holdout test setini federated için bırak
    test_df = X_test.copy()
    test_df[label_col] = y_test.values
    holdout_path = FEDERATED_DIR / f"holdout_test_{label_col}.csv"
    test_df.to_csv(holdout_path, index=False)
    print(f"  ✓ Holdout test: {holdout_path}")
    print()

FINAL MODEL EĞİTİMİ
Feature sayısı: 45

--- label_multiclass ---
  Accuracy    : 0.9960
  F1 Weighted : 0.9959
  F1 Macro    : 0.9068
  ✓ Kaydedildi: ../models/rf_multiclass.pkl
  ✓ Holdout test: ../federated/holdout_test_label_multiclass.csv

--- label_binary ---
  Accuracy    : 0.9978
  F1 Weighted : 0.9978
  F1 Macro    : 0.9968
  ✓ Kaydedildi: ../models/rf_binary.pkl
  ✓ Holdout test: ../federated/holdout_test_label_binary.csv

